## 🧠 What is Query Decomposition?
Query decomposition（查询分解） is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

In [3]:
## step 1: Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
# Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)

# Step 3: MMR Retriever
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "lambda_mult": 0.7})
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002258A473A10>, search_type='mmr', search_kwargs={'k': 5, 'lambda_mult': 0.7})

In [5]:
# Step 4: LLM and Prompt

import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.4,
    api_key=api_key,
    base_url=base_url
)
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002258CB4F770>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002258DFC0440>, root_client=<openai.OpenAI object at 0x000002258CB4C980>, root_async_client=<openai.AsyncOpenAI object at 0x000002258DFC01A0>, temperature=0.4, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1')

In [7]:
# Step 5: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()
decomposition_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.\n\nQuestion: "{question}"\n\nSub-questions:\n')
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002258CB4F770>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002258DFC0440>, root_client=<openai.OpenAI object at 0x000002258CB4C980>,

In [9]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question": query})
print(decomposition_question)

Sub-questions:  
1. What types of memory does LangChain use and how are they implemented?  
2. How does LangChain utilize agents in its framework?  
3. What types of memory and agent mechanisms does CrewAI use?  
4. How do the memory and agent approaches of LangChain and CrewAI differ from each other?


In [10]:
# Step 6: QA chain per sub-question
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
qa_chain = create_retrieval_chain(retriever, qa_prompt)
qa_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002258A473A10>, search_type='mmr', search_kwargs={'k': 5, 'lambda_mult': 0.7}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {input}\n')
  }), kwargs={}, config={'run_name': 'retrieval_chain'}, config_factories=[])

In [17]:
rag_chain = create_retrieval_chain(retriever, qa_prompt)

In [18]:
def full_query_decomposition_rag_pipeline(user_query):
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-.1234567890") for q in sub_qs_text.split("\n") if q.strip()]

    results = []
    for subq in sub_questions:
        result = rag_chain.invoke({"input": subq})
        results.append(f"Q: {subq}\nA: {result}")
    return "\n\n".join(results)

In [19]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: Sub-questions:  
A: {'input': 'Sub-questions:  ', 'context': [Document(id='158c7760-496f-425e-8417-752eb68702aa', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content="One of CrewAI's core innovations is the use of agent context-sharing, where agents pass intermediate data to one another in a structured manner. This leads to emergent behaviors like delegation, consultation, and review among agents. (v3)"), Document(id='7d939bf9-8ed6-4c02-b35b-776f60924a5e', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)'), Document(id='d27b05aa-1c0b-4227-b5f9-2151384cac49', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='knowledge is fetched and injected into the LLM prompt to enhance accuracy and reduce hallucination. (v1)'), Document(id='715ae873-12a6-4980-a88f-7acc7f9eab3d', metadata={'source': 'langchain_crewai_dataset.txt'}, page_